# 21 — Family 1 (F1): Two-Stage Anchor-Based Detector (`Faster R-CNN R50-FPN v2`)

**Experiments under Fixed Protocol P0 (`k5_drop_L1920`, `1.5 GPU-h`, `max_size=1280`):**
- **`frcnn-e0` (Baseline):** COCO-pretrained `fasterrcnn_resnet50_fpn_v2`, `min_size=720, max_size=1280`, `box_detections_per_img=500`, `box_score_thresh=0.001`.
- **`frcnn-e1` (Exp A — Proposal & Sampling Density):** RPN proposals expanded (`4000/2000` train, `2000/2000` test), RoI batch size `1024`, `positive_fraction=0.33`, and halved RPN base anchors (`16..256 px`).
- **`frcnn-e2` (Exp B — GIoU RoI Box Loss):** Replacing Smooth-L1 RoI regression loss with scale-invariant GIoU loss on decoded proposals (`tv_patches.giou_fastrcnn_loss`).

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from vdet.paths import P
from vdet.utils import seed_everything, load_json
from vdet.tracking.plots import curves, size_breakdown
seed_everything(42)

In [ ]:
# 1. Training commands (run in terminal or Kaggle/Colab cell):
# !python scripts/train_tv.py --config configs/faster_rcnn/e0_base.yaml --eval-val
# !python scripts/train_tv.py --config configs/faster_rcnn/e1_rpn_proposals.yaml --eval-val
# !python scripts/train_tv.py --config configs/faster_rcnn/e2_giou.yaml --eval-val
# !python scripts/benchmark_speed.py --weights checkpoints/frcnn-e0_best.pth --run-id frcnn-e0

In [ ]:
# 2. Display F1 results table from experiments/results.csv and RPN proposal recall diagnostics
if P.results_csv.exists():
    df = pd.read_csv(P.results_csv)
    display(df[df['family'] == 'faster_rcnn'][['run_id', 'exp', 'AP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR500', 'fps', 'gpu_h', 'epochs_done']])
for rid in ['frcnn-e0', 'frcnn-e1', 'frcnn-e2']:
    rpn_p = P.exp_runs_dir / rid / 'rpn_recall.json'
    if rpn_p.exists():
        print(rid, 'RPN recall:', load_json(rpn_p))